In [0]:
%pip install seaborn sqlalchemy

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sqlalchemy import create_engine, text

In [0]:
import pandas as pd

# Identifico la ruta del archivo de restaurantes
url_restaurante = "/Workspace/Users/bances27caro@gmail.com/CURSO_PHYTON/Trabajo-Final-Python/restaurants.csv/restaurants.csv"

# Cargo el archivo
df_restaurantes = pd.read_csv(url_restaurante)

# Visualizó las primeras filas
display(df_restaurantes.head())

In [0]:
import pandas as pd

# Ruta de la carpeta donde están las partes de los menús
ruta_menus = "/Workspace/Users/bances27caro@gmail.com/CURSO_PHYTON/Trabajo-Final-Python/restaurant-menus.csv"

# Lista con los nombres de los 10 archivos
archivos_menus = [
    f"restaurant_menus_parte_{i}.csv"
    for i in range(1, 11)
]

# Leemos cada archivo y lo guardamos en una lista
lista_menus = []

for archivo in archivos_menus:
    ruta = f"{ruta_menus}/{archivo}"
    df_parte = pd.read_csv(ruta)
    lista_menus.append(df_parte)

# Unimos todas las partes
df_menus = pd.concat(lista_menus, ignore_index=True)

# Mostramos las primeras filas
display(df_menus.head())

In [0]:
# Dimensiones del DataFrame de restaurantes
print(f"df_restaurantes: contiene {df_restaurantes.shape[0]} filas y {df_restaurantes.shape[1]} columnas")

# Dimensiones del DataFrame de menús
print(f"df_menus: contiene {df_menus.shape[0]} filas y {df_menus.shape[1]} columnas")

In [0]:
# Perfil de datos de restaurantes
perfil_restaurantes = pd.DataFrame({
    "Categoria": df_restaurantes.columns,
    "Tipo de dato": df_restaurantes.dtypes.astype(str).values,
    "Valores nulos": df_restaurantes.isna().sum().values
})

display(perfil_restaurantes)

In [0]:
# Perfil de datos de menús
perfil_menus = pd.DataFrame({
    "Categoria": df_menus.columns,
    "Tipo de dato": df_menus.dtypes.astype(str).values,
    "Valores nulos": df_menus.isna().sum().values
})

display(perfil_menus)

In [0]:
mapeo_precios = {
    "$": "Económico",
    "$$": "Moderadamente caro",
    "$$$": "Caro",
    "$$$$": "Muy caro"
}

# Se determina la cantidad de símbolos
df_restaurantes["rango_de_precios"] = df_restaurantes["price_range"].map(mapeo_precios)
df_restaurantes["rango_de_precios"].value_counts()

In [0]:
display(df_restaurantes[["full_address"]].head())

In [0]:
# Se utiliza .str.split() para dividir la columna "full_address" en tres columnas
df_restaurantes[["calle", "ciudad", "estado_zip"]] = (
    df_restaurantes["full_address"].str.split(",", n=2, expand=True)
)
# Se elimina los espacios que puedan quedar antes o después del nombre de la ciudad.
df_restaurantes["ciudad"] = df_restaurantes["ciudad"].str.strip()

display(df_restaurantes[["calle", "ciudad", "estado_zip"]].head())

In [0]:
# Eliminamos los score nulos
df_restaurantes = df_restaurantes[df_restaurantes["score"].notna()]

# Eliminamos los score iguales a 0
df_restaurantes = df_restaurantes[df_restaurantes["score"] != 0]

# Estandarizamos category
df_restaurantes["category"] = (df_restaurantes["category"].str.lower())

display(df_restaurantes[["score", "category"]].head(10))


In [0]:
print("Valores nulos en score:", df_restaurantes["score"].isna().sum())
print("Scores iguales a 0:", (df_restaurantes["score"] == 0).sum())

In [0]:
# Eliminamos los ítems del menú que no tienen precio registrado
df_menus = df_menus[df_menus["price"].notna()]

# Eliminamos el texto " USD"
df_menus["price"] = df_menus["price"].astype(str).str.replace(" USD", "", regex=False)

# Reemplazamos la coma decimal por un punto
df_menus["price"] = df_menus["price"].str.replace(",", ".", regex=False)

# Convertimos la columna a formato decimal
df_menus["price"] = df_menus["price"].astype(float)

# Visualizamos el resultado de la transformación
display(df_menus[["price"]].head(10))

In [0]:
print("Precios nulos:", df_menus["price"].isna().sum())
print("Precios iguales a 0:", (df_menus["price"] == 0).sum())

In [0]:
# Eliminamos los precios nulos
df_menus = df_menus[df_menus["price"].notna()]

# Eliminamos los precios iguales a 0
df_menus = df_menus[df_menus["price"] != 0]

# Visualizamos los datos después de la limpieza
display(df_menus[["price"]].head(10))

In [0]:
print("Precios nulos:", df_menus["price"].isna().sum())
print("Precios iguales a 0:", (df_menus["price"] == 0).sum())

In [0]:
# Visualizamos la columna de calificaciones
display(df_restaurantes[["ratings"]].head(10))

# Filtramos los restaurantes que tienen más de 100 calificaciones
df_restaurantes = df_restaurantes[df_restaurantes["ratings"] > 100]

# Visualizamos el resultado del filtro
display(df_restaurantes[["ratings"]].head(10))

In [0]:
print("Cantidad de restaurantes después del filtro:", len(df_restaurantes))
print("Cantidad mínima de ratings:",df_restaurantes["ratings"].min())

In [0]:
# Realizamos el INNER JOIN mediante el ID del restaurante
df_master = pd.merge(
    df_restaurantes,
    df_menus,
    left_on="id",
    right_on="restaurant_id",
    how="inner"
)

# Visualizamos el DataFrame resultante
display(df_master.head())

In [0]:
# Creamos una copia del DataFrame resultante del INNER JOIN
df_final = df_master.copy()

# Revisamos las columnas disponibles antes de realizar la limpieza
print("Columnas iniciales:")
print(df_final.columns.tolist())

# Eliminamos las columnas que ya no son necesarias
df_final = df_final.drop(
    columns=["restaurant_id", "full_address"]
)

# Cambiamos los nombres para identificar mejor la información
df_final = df_final.rename(
    columns={
        "name_x": "nombre_restaurante",
        "category_x": "categoria_restaurante",
        "name_y": "nombre_plato",
        "category_y": "categoria_menu"
    }
)

# Visualizamos el DataFrame después de la limpieza
display(df_final.head())